In [2]:
!pip install scikeras

# Uninstall the current scikit-learn version
!pip uninstall scikit-learn -y

# Install a compatible version of scikit-learn (e.g., 1.4.2)
!pip install scikit-learn==1.4.2

Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 85.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from keras.callbacks import EarlyStopping

from scikeras.wrappers import KerasClassifier

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier


In [5]:
# Read training data
train_df = pd.read_csv('train.csv')
train_df

,match_id,color,rank,map_code,duration,car_name,possession_time,time_in_side,shots,shots_against,...,percent_defensive_half,percent_offensive_half,percent_behind_ball,percent_infront_ball,percent_most_back,percent_most_forward,percent_closest_to_ball,percent_farthest_from_ball,demos_inflicted,demos_taken
0,0,blue,silver,neotokyo_standard_p,163,Breakout,54.95,58.96,2,4,...,62.736664,37.263340,66.842890,33.157120,95.534080,95.534080,95.534080,95.534080,1,0
1,0,orange,silver,neotokyo_standard_p,163,Octane,27.51,80.68,4,2,...,63.576576,36.423428,77.846670,22.153326,98.304170,98.304170,98.304170,98.304170,0,1
2,1,blue,gold,stadium_foggy_p,460,Octane,132.98,244.72,7,10,...,75.199120,24.800879,73.992320,26.007679,96.942440,96.942440,96.942440,96.942440,0,0
3,1,orange,gold,stadium_foggy_p,460,Octane,102.55,148.84,10,7,...,55.832005,44.167990,77.792800,22.207201,96.918510,96.918510,96.918510,96.918510,0,0
4,2,blue,silver,neotokyo_standard_p,94,Octane,25.24,33.70,0,3,...,76.376495,23.623503,68.454840,31.545156,93.636470,93.636470,93.636470,93.636470,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60237,30118,orange,silver,chn_stadium_p,390,Ronin GXT,94.20,193.21,8,6,...,64.023270,35.976734,72.403404,27.596594,98.048460,98.048460,98.048460,98.048460,0,1
60238,30119,blue,silver,trainstation_p,372,Fennec,145.58,123.44,8,3,...,56.082024,43.917976,78.380820,21.619183,98.296730,98.296730,98.296730,98.296730,0,2
60239,30119,orange,silver,trainstation_p,372,Octane,119.14,213.29,3,8,...,72.444016,27.555986,70.517235,29.482769,100.418655,100.418655,100.418655,100.418655,2,0
60240,30120,blue,platinum,wasteland_night_s_p,425,Octane,131.41,184.57,7,7,...,58.544390,41.455605,73.541850,26.458149,97.486570,97.486570,97.486570,97.486570,1,0


In [75]:
# Get decision tree feature importance
drop_columns = ['match_id', 'rank', 'color', 'car_name', 'map_code', 'assists', 'mvp', 'demos_inflicted', 'demos_taken']
match_df = train_df.groupby(['match_id', 'rank']).sum().reset_index()

X = match_df.drop(columns=drop_columns)
y = match_df['rank']

X_tree_train, X_tree_test, y_tree_train, y_tree_test = train_test_split(
    X, y,
    test_size = 0.25,
    random_state = 42,
    stratify=y
)

ct_tree = ColumnTransformer(
    transformers=[
        ('imputer', SimpleImputer(), ['possession_time', 'time_in_side'])
    ],
    remainder='passthrough'
)

pipe_tree = Pipeline(
    steps = [
        ('transformer', ct_tree),
        ('scaler', StandardScaler()),
        ('model', RandomForestClassifier(max_depth=5))
    ]
).fit(X_tree_train, y_tree_train)

y_pred_tree_train = pipe_tree.predict(X_tree_train)
y_pred_tree_test = pipe_tree.predict(X_tree_test)

print(classification_report(
    y_true=y_tree_train,
    y_pred=y_pred_tree_train
))
print(classification_report(
    y_true=y_tree_test,
    y_pred=y_pred_tree_test
))

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

      bronze       0.00      0.00      0.00       553
    champion       0.66      0.70      0.68      4408
     diamond       0.48      0.41      0.44      5187
        gold       0.49      0.54      0.51      4689
    platinum       0.45      0.53      0.49      5623
      silver       0.53      0.40      0.46      2130

    accuracy                           0.51     22590
   macro avg       0.43      0.43      0.43     22590
weighted avg       0.50      0.51      0.51     22590

              precision    recall  f1-score   support

      bronze       0.00      0.00      0.00       184
    champion       0.63      0.66      0.64      1470
     diamond       0.44      0.39      0.41      1729
        gold       0.47      0.51      0.49      1563
    platinum       0.43      0.50      0.46      1875
      silver       0.52      0.39      0.44       710

    accuracy                           0.49      7531
   macro avg       0.41

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [76]:
features_df = pd.DataFrame(
    data={
        'feature': pipe_tree[:-1].get_feature_names_out(),
        'importance': pipe_tree.named_steps['model'].feature_importances_
    }
)
important_features_df = features_df.sort_values('importance')
important_features_df.head(10)

,feature,importance
59,remainder__time_defensive_third,0.000035
20,remainder__count_stolen_big,0.000048
71,remainder__percent_defensive_third,0.000098
68,remainder__goals_against_while_last_defender,0.000112
23,remainder__amount_overfill,0.000116
16,remainder__amount_stolen_big,0.000118
5,remainder__goals,0.000119
67,remainder__time_most_forward,0.000124
73,remainder__percent_neutral_third,0.000130
18,remainder__amount_stolen_small,0.000150


In [77]:
le = LabelEncoder()
y = le.fit_transform(y)
le.classes_

array(['bronze', 'champion', 'diamond', 'gold', 'platinum', 'silver'],
      dtype=object)

In [78]:
# Split train and test data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [81]:
# Initialize column transformer
ct = ColumnTransformer(
    transformers=[
        # ('ohe', OneHotEncoder(), ['map_code']),
        ('imputer', SimpleImputer(), X.columns)
    ],
    remainder='passthrough'
)

In [83]:
es = EarlyStopping(monitor='val_loss', patience=5)

In [84]:
# Function to create the Keras model for SciKeras
def create_model():
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.InputLayer(shape=(len(X.columns),)))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(6, activation='softmax')) # Need output node per rank
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Keras model with SciKeras wrapper
model = KerasClassifier(
    model=create_model,
    epochs=100,
    validation_split=0.2,
    callbacks=[es]
)

In [85]:
# Create and fit model pipeline
pipe = Pipeline(
    steps=[
        ('transformer', ct),
        ('scaler', StandardScaler()),
        ('model', model)
    ]
).fit(X_train, y_train)

Epoch 1/100
603/603 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.5226 - loss: 1.0797 - val_accuracy: 0.5313 - val_loss: 1.0283
Epoch 2/100
603/603 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5534 - loss: 0.9953 - val_accuracy: 0.5631 - val_loss: 0.9890
Epoch 3/100
603/603 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.5672 - loss: 0.9712 - val_accuracy: 0.5546 - val_loss: 0.9912
Epoch 4/100
603/603 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.5742 - loss: 0.9579 - val_accuracy: 0.5612 - val_loss: 0.9950
Epoch 5/100
603/603 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5744 - loss: 0.9508 - val_accuracy: 0.5558 - val_loss: 0.9900
Epoch 6/100
603/603 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5838 - loss: 0.9362 - val_accuracy: 0.5618 - val_loss: 1.0030
Epoch 7/100
603/603 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5881 - loss: 0.9258 - val_accuracy: 0.5552 - val_loss: 1.0105


In [86]:
y_pred_train = pipe.predict(X_train)
y_pred_test = pipe.predict(X_test)

753/753 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
189/189 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [ ]:
confusion_matrix(
    y_true=y_train,
    y_pred=y_pred_train
)

array([[ 216,    1,    4,   83,   19,  193],
       [   1, 3049,  912,    8,  144,    1],
       [   5, 1030, 2468,  125, 1211,    2],
       [  30,    8,  118, 2956,  962,  302],
       [  10,  140,  996, 1272, 2807,   23],
       [  98,    2,    3,  893,   61,  931]])

In [ ]:
confusion_matrix(
    y_true=y_test,
    y_pred=y_pred_test
)

array([[  71,    0,    0,   52,   11,   87],
       [   0, 1248,  443,    2,   70,    0],
       [   1,  507,  961,   59,  547,    0],
       [  11,    5,   50, 1177,  493,  140],
       [   3,   66,  462,  578, 1129,   12],
       [  50,    1,    3,  415,   31,  352]])

In [87]:
print(classification_report(
    y_true=y_train,
    y_pred=y_pred_train
))
print(classification_report(
    y_true=y_test,
    y_pred=y_pred_test
))

              precision    recall  f1-score   support

           0       0.74      0.25      0.37       590
           1       0.82      0.59      0.69      4702
           2       0.52      0.66      0.58      5533
           3       0.54      0.70      0.61      5001
           4       0.55      0.45      0.50      5998
           5       0.61      0.59      0.60      2272

    accuracy                           0.59     24096
   macro avg       0.63      0.54      0.56     24096
weighted avg       0.61      0.59      0.58     24096

              precision    recall  f1-score   support

           0       0.76      0.20      0.31       147
           1       0.76      0.53      0.63      1176
           2       0.47      0.60      0.53      1383
           3       0.50      0.66      0.57      1251
           4       0.50      0.41      0.45      1500
           5       0.57      0.54      0.55       568

    accuracy                           0.54      6025
   macro avg       0.60

In [88]:
test_df = pd.read_csv('test.csv')

In [89]:
match_test_df = test_df.groupby(['match_id']).sum().reset_index()

In [92]:
y_pred = pipe.predict(match_test_df[X.columns])

79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [93]:
le.classes_

array(['bronze', 'champion', 'diamond', 'gold', 'platinum', 'silver'],
      dtype=object)

In [94]:
converter = { 0: 1, 5: 2, 3: 3, 4: 4, 2: 5, 1: 6 }

y_pred = pd.Series(y_pred).map(converter)

In [95]:
submission = pd.concat([match_test_df['match_id'], y_pred], axis = 1).rename(columns = {0: 'rank'})
submission.head()

,match_id,rank
0,30121,5
1,30122,2
2,30123,4
3,30124,4
4,30125,5


In [98]:
submission.to_csv('keras_important_features_3-128_relu_submission.csv', index=False)

In [97]:
submission.shape

(2500, 2)